# MASLD-Bench · 02 — Classical ML Baselines + Frontier LLM Evaluation

Two goals, mirroring the paper's evaluation core:

1. **Classical baselines** trained on the exact features each prompt exposes (ElasticNet / HistGradientBoosting / XGBoost fibrosis clocks). These are the numbers any LLM must beat.
2. **Frontier LLM evaluation** on the frozen test JSONLs, via OpenRouter, zero-shot greedy single-pass — the paper's protocol.

Runs on a **T4 / CPU runtime** (no GPU needed for baselines; API calls are network-bound).

Outputs: `results/baselines.json`, `results/frontier_runs/<model>.jsonl.gz`, `results/leaderboard.csv`

## 0 · Setup

In [ ]:
from google.colab import drivedrive.mount('/content/drive')MASLD_REV = '/content/drive/MyDrive/masld/data/masld-revision'PROMPTS   = '/content/drive/MyDrive/masld/prompts'RESULTS   = '/content/drive/MyDrive/masld/results'import os, json, gzip, time, reimport numpy as np, pandas as pdfrom sklearn.linear_model import ElasticNetCV, LogisticRegressionfrom sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifierfrom sklearn.metrics import concordance_index, f1_score, mean_absolute_erroros.makedirs(RESULTS, exist_ok=True)os.makedirs(f'{RESULTS}/frontier_runs', exist_ok=True)

## 1 · Load the frozen prompts

In [ ]:
def load_jsonl(path):    with gzip.open(path,'rt') as f:        first = json.loads(f.readline())        meta = first.get('__meta__', {}) if '__meta__' in first else {}        rows = [json.loads(l) for l in f] if '__meta__' in first else [first] + [json.loads(l) for l in f]    return meta, pd.DataFrame(rows)for fn in sorted(os.listdir(PROMPTS)):    if fn.endswith('.jsonl.gz'):        m, df = load_jsonl(os.path.join(PROMPTS, fn))        print(f"{fn:40s} {df.shape[0]:6d} prompts  tasks={sorted(df.task.unique())}")

## 2 · Classical baselines — fibrosis clocks on prompt features
Paper rule: *every ML model was trained only on the features that the corresponding prompt exposed to the LLMs, and evaluated on the identical frozen test items.* We parse the `gene=value` tokens back out of the prompts to build X, so the comparison is exactly paired.

In [ ]:
def parse_features(prompt):    feats = {}    for m in re.finditer(r'([A-Za-z0-9_.\-]+)=([0-9.]+)', prompt):        feats[m.group(1)] = float(m.group(2))    return featsdef build_xy(df):    X = df['prompt'].map(parse_features)    all_genes = sorted({g for d in X for g in d})    mat = np.zeros((len(df), len(all_genes)))    gene_ix = {g:i for i,g in enumerate(all_genes)}    for r, d in enumerate(X):        for g,v in d.items(): mat[r, gene_ix[g]] = v    return mat, all_genes# --- regression: fibrosis stage 0-4 ---def fit_clock(train_df, test_df, target, model='en'):    Xtr, genes = build_xy(train_df)    Xte, _     = build_xy(test_df)    # align columns    gene_ix = {g:i for i,g in enumerate(genes)}    Xte2 = np.zeros_like(Xte[:, :len(genes)]) if False else None    ytr = pd.to_numeric(train_df[target], errors='coerce')    yte = pd.to_numeric(test_df[target], errors='coerce')    keep = ytr.notna(); Xtr, ytr = Xtr[keep], ytr[keep]    if model == 'en':        reg = ElasticNetCV(l1_ratio=[.1,.5,.9], cv=5, n_jobs=-1).fit(Xtr, ytr)    else:        reg = HistGradientBoostingRegressor().fit(Xtr, ytr)    pred = reg.predict(Xte)    return dict(cindex=concordance_index(yte, pred),                mae=mean_absolute_error(yte, np.clip(pred,0,4).round()))

## 3 · Frontier LLM evaluation — OpenRouter, zero-shot greedy single-pass
Sign up at openrouter.ai, buy ~$10 credit, create a key. Store it in Colab secrets (the key icon on the left).

Model menu (paper's roster translated to OpenRouter ids — pick by budget):

| Tier | OpenRouter id | Why |
|---|---|---|
| must | `deepseek/deepseek-chat` | cheap, paper's mid-roster |
| must | `google/gemini-2.5-pro` | paper's top frontier model lineage |
| must | `openai/gpt-4.1-mini` | price/perf |
| nice | `anthropic/claude-sonnet-4.5`, `x-ai/grok-4-fast`, `moonshotai/kimi-k2` | coverage |

Cost control: the paper ran **25,457 prompts**; you don't need to. Stratify to ~300–500 prompts per task first (`df.sample(n=min(500,len(df)), random_state=...)`).

In [ ]:
!pip -q install openaifrom google.colab import userdatafrom openai import OpenAIclient = OpenAI(    base_url='https://openrouter.ai/api/v1',    api_key=userdata.get('OPENROUTER_API_KEY'),   # add in Colab secrets)MODEL = 'deepseek/deepseek-chat'   # start cheap; then gemini-2.5-pro, gpt-4.1-minidef ask_llm(prompt, max_tokens=1500):    r = client.chat.completions.create(        model=MODEL,        messages=[{'role':'user','content':prompt}],        temperature=0.0, top_p=1.0, max_tokens=max_tokens)    return r.choices[0].message.content# --- validity parsing (paper: invalid answers counted, response-validity reported) ---def parse_answer(text, task):    t = (text or '').strip().lower()    if task.endswith(('MB-1001','MB-2001')):            # binary        return 'yes' if 'yes' in t else 'no' if 'no' in t else None    if task.endswith(('MB-1003','MB-2003')):            # pairwise        if t.startswith('a'): return 'A'        if t.startswith('b'): return 'B'        return None    m = re.search(r'f([0-4])\b', t)                     # multiclass F0-F4    if m: return f'F{m.group(1)}'    m = re.search(r'[0-9]+\.?[0-9]*', t)               # regression    return float(m.group()) if m else None

In [ ]:
def evaluate_frontier(jsonl_path, n_max=500):    meta, df = load_jsonl(jsonl_path)    df = df[df.split=='test'] if 'split' in df else df    df = df.sample(n=min(n_max, len(df)), random_state=20260929)    out_path = f"{RESULTS}/frontier_runs/{MODEL.replace('/','_')}__{os.path.basename(jsonl_path)}"    with gzip.open(out_path, 'wt') as f:        for i, row in enumerate(df.itertuples()):            try:                raw = ask_llm(row.prompt)            except Exception as e:                raw = f'__error__ {e}'            rec = dict(task=row.task, sample=row.sample, answer=row.answer,                       raw=raw, parsed=parse_answer(raw, row.task))            f.write(json.dumps(rec)+'\n')            if (i+1) % 25 == 0: print(f'{i+1}/{len(df)}')    return out_path# run on each frozen test filefor fn in ['MB-2xxx_liver_test.jsonl.gz']:   # extend list as builders land    if os.path.exists(f'{PROMPTS}/{fn}'):        evaluate_frontier(f'{PROMPTS}/{fn}')

## 4 · Score & leaderboard
Paper metrics: concordance for pairwise, accuracy/F1 for classification, MAE + Spearman for regression; Friedman + post-hoc McNemar/Wilcoxon with Holm correction. On your prompt counts, report bootstrap CIs instead of Friedman critical-difference (that needs many models×tasks).

In [ ]:
from scipy.stats import spearmanrdef score_run(path):    with gzip.open(path,'rt') as f:        rows = [json.loads(l) for l in f]    df = pd.DataFrame(rows)    s = {}    for task, g in df.groupby('task'):        valid = g.parsed.notna()        if g.answer.iloc[0] in ('A','B'):            s[task] = dict(metric='concordance',                           value=float((g.parsed==g.answer).mean()),                           validity=float(valid.mean()))        elif str(g.answer.iloc[0]).startswith('F'):            s[task] = dict(metric='accuracy',                           value=float((g.parsed.astype(str)==g.answer.astype(str))[valid].mean()),                           validity=float(valid.mean()))        else:            num = pd.to_numeric(g.parsed, errors='coerce')            s[task] = dict(metric='mae',                           value=float(np.abs(num-g.answer.astype(float))[num.notna()].mean()),                           validity=float(valid.mean()))    return s# assemble leaderboard rows -> results/leaderboard.csvprint('baseline + frontier scoring wired; extend as runs land')